In [2]:
import dash
import plotly.express as px
import plotly.graph_objects as go
import dash_bootstrap_components as dbc
import pandas as pd
from dash import html, dcc, Input, Output
import plotly.io as pio

# Tema obscuro
pio.templates.default = "plotly_dark"

# =========================
df = pd.read_csv('./clinical_analytics.csv')

In [3]:
df['Wait Time Min'] = pd.to_numeric(df['Wait Time Min'], errors='coerce')
df['Care Score'] = pd.to_numeric(df['Care Score'], errors='coerce')
df = df.dropna(subset=['Wait Time Min', 'Care Score'])

df['Check-In Time'] = pd.to_datetime(
    df['Check-In Time'],
    format='%Y-%m-%d %I:%M:%S %p',
    errors='coerce'
)

df['Check-In Month'] = df['Check-In Time'].dt.to_period('M')
df['Admit Source'] = df['Admit Source'].fillna('Unknown')

df = df[df['Wait Time Min'] >= 0]

In [4]:
# COLORES
BG = "#2b2b2b"
CARD = "#3a3a3a"
BLUE = "#4dabf7"
TEXT = "#a5d8ff"

# APP
app = dash.Dash(external_stylesheets=[dbc.themes.DARKLY])

def kpi_card(title, id_value):
    return dbc.Card(
        dbc.CardBody([
            html.P(title, style={"color": TEXT}),
            html.H3(id=id_value, style={"color": BLUE}),
        ]),
        style={
            "backgroundColor": CARD,
            "borderRadius": "10px"
        }
    )

In [5]:
# LAYOUT

app.layout = dbc.Container(fluid=True, children=[

    dbc.Row([

        # SIDEBAR
        dbc.Col([
            html.H4("🏥 Clínica", style={"color": BLUE}),
            html.Hr(),

            html.Label("Departamento", style={"color": TEXT}),
            dcc.Dropdown(
                id="department-filter",
                options=[{'label': x, 'value': x} for x in df['Department'].unique()],
                value=df['Department'].unique(),
                multi=True
            ),

            html.Br(),

            html.Label("Tipo Admisión", style={"color": TEXT}),
            dcc.Dropdown(
                id="admit-type-filter",
                options=[{'label': x, 'value': x} for x in df['Admit Type'].unique()],
                value=df['Admit Type'].unique(),
                multi=True
            ),

            html.Br(),

            html.Label("Fuente", style={"color": TEXT}),
            dcc.Dropdown(
                id="admit-source-filter",
                options=[{'label': x, 'value': x} for x in df['Admit Source'].unique()],
                value=df['Admit Source'].unique(),
                multi=True
            ),

            html.Br(),

            html.Label("Tiempo Espera", style={"color": TEXT}),
            dcc.RangeSlider(
                id="wait-filter",
                min=df['Wait Time Min'].min(),
                max=df['Wait Time Min'].max(),
                value=[df['Wait Time Min'].min(), df['Wait Time Min'].max()],
            ),

        ], width=2, style={
            "backgroundColor": "#1f1f1f",
            "height": "100vh",
            "padding": "15px",
            "position": "fixed"
        }),

        # CONTENIDO
        dbc.Col([

            html.H2("Dashboard Clínico", style={"color": BLUE}),
            html.P("Análisis de operaciones", style={"color": TEXT}),

            # KPIs
            dbc.Row([
                dbc.Col(kpi_card("Tiempo Promedio", "avg-wait"), md=3),
                dbc.Col(kpi_card("Care Score", "avg-care"), md=3),
                dbc.Col(kpi_card("Registros", "total-rec"), md=3),
                dbc.Col(kpi_card("Clínicas", "clinics"), md=3),
            ], className="mb-4"),

            # FILA 1
            dbc.Row([
                dbc.Col(dcc.Graph(id="scatter"), md=6),
                dbc.Col(dcc.Graph(id="bar-dept"), md=6),
            ]),

            # FILA 2
            dbc.Row([
                dbc.Col(dcc.Graph(id="hist"), md=6),
                dbc.Col(dcc.Graph(id="timeline"), md=6),
            ]),

            # FILA 3
            dbc.Row([
                dbc.Col(dcc.Graph(id="dual-bar"), md=12),
            ]),

            # FILA 4
            dbc.Row([
                dbc.Col(dcc.Graph(id="box"), md=6),
                dbc.Col(dcc.Graph(id="heatmap"), md=6),
            ]),

            # FILA 5
            dbc.Row([
                dbc.Col(dcc.Graph(id="clinic-bar"), md=6),
                dbc.Col(dcc.Graph(id="status-bar"), md=6),
            ]),

            dbc.Row([
                dbc.Col(html.Div(id="insight-box", style={"color": TEXT}))
            ])

        ], width={"size": 10, "offset": 2}, style={
            "backgroundColor": BG,
            "padding": "20px",
            "minHeight": "100vh"
        })

    ])
])

In [6]:
#  CALLBACK

@app.callback(
    [
        Output("scatter", "figure"),
        Output("bar-dept", "figure"),
        Output("hist", "figure"),
        Output("timeline", "figure"),
        Output("dual-bar", "figure"),
        Output("box", "figure"),
        Output("heatmap", "figure"),
        Output("clinic-bar", "figure"),
        Output("status-bar", "figure"),
        Output("avg-wait", "children"),
        Output("avg-care", "children"),
        Output("total-rec", "children"),
        Output("clinics", "children"),
        Output("insight-box", "children"),
    ],
    [
        Input("department-filter", "value"),
        Input("admit-type-filter", "value"),
        Input("admit-source-filter", "value"),
        Input("wait-filter", "value"),
    ]
)
def update(dept, admit, source, wait):

    dff = df[
        (df['Department'].isin(dept)) &
        (df['Admit Type'].isin(admit)) &
        (df['Admit Source'].isin(source)) &
        (df['Wait Time Min'].between(wait[0], wait[1]))
    ]

    #  Sinteti
    if dff.empty:
        empty = go.Figure()
        return [empty]*9 + ["0","0","0","0","Sin datos"]

    # SCATTER
    scatter = px.scatter(dff, x="Wait Time Min", y="Care Score", color="Department")

    # BAR
    top_dept = dff['Department'].value_counts().nlargest(10).reset_index()
    top_dept.columns = ['Department', 'Count']
    bar_dept = px.bar(top_dept, x="Count", y="Department", orientation="h")

    # HIST
    hist = px.histogram(dff, x="Wait Time Min", color="Encounter Status")

    # TIMELINE
    timeline_df = dff.groupby('Check-In Month').size().reset_index(name='Cantidad')
    timeline_df['Check-In Month'] = timeline_df['Check-In Month'].astype(str)
    timeline = px.line(timeline_df, x="Check-In Month", y="Cantidad")

    # DUAL
    stats = dff.groupby('Admit Source').agg({
        'Wait Time Min':'mean',
        'Encounter Number':'count'
    }).reset_index()

    dual = go.Figure()
    dual.add_trace(go.Bar(x=stats['Admit Source'], y=stats['Encounter Number']))
    dual.add_trace(go.Scatter(x=stats['Admit Source'], y=stats['Wait Time Min'], yaxis="y2"))
    dual.update_layout(yaxis2=dict(overlaying='y', side='right'))

    # BOX
    box = px.box(dff, x="Department", y="Wait Time Min")

    # HEATMAP
    corr = dff[['Wait Time Min','Care Score']].corr()
    heatmap = px.imshow(corr, text_auto=True)

    # CLINIC BAR
    clinic = dff.groupby('Clinic Name')['Wait Time Min'].mean().reset_index()
    clinic = clinic.sort_values(by='Wait Time Min', ascending=False).head(10)
    clinic_bar = px.bar(clinic, x="Wait Time Min", y="Clinic Name", orientation="h")

    # STATUS
    status_bar = px.histogram(dff, x="Department", color="Encounter Status")

    # KPIs
    avg_wait = f"{dff['Wait Time Min'].mean():.1f}"
    avg_care = f"{dff['Care Score'].mean():.2f}"
    total = f"{len(dff):,}"
    clinics = f"{dff['Clinic Name'].nunique()}"

    insight = "Dashboard funcionando correctamente ✔"

    return (
        scatter, bar_dept, hist, timeline, dual,
        box, heatmap, clinic_bar, status_bar,
        avg_wait, avg_care, total, clinics, insight
    )

In [8]:
# =========================
# RUN
# =========================
if __name__ == "__main__":
    app.run(debug=True, port=8060)